# 05 — Wage Stagnation & The Worker Subsidy

**Goal:** Show that the $10.60/hr wage assumption has been frozen while inflation eroded its
purchasing power, and quantify the collective "subsidy" that direct care workers provide
to the Texas Medicaid system through below-market labor.

**Data Sources:**
- BLS CPI-U (South Urban) via API — inflation since the wage was set
- BLS OEWS May 2024 — current market wages and employment count
- PFD Wage Calculator — the $10.60 assumption
- Market wage comparisons from notebook 03

**Key Questions:**
1. What is $10.60 worth in real (inflation-adjusted) dollars today?
2. How much do workers collectively lose by accepting Medicaid-level wages vs. market alternatives?
3. What does a career in direct care cost someone over a lifetime vs. retail?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from texas_hhcs.cpi import fetch_cpi_series, annual_cpi, deflate_wage, build_erosion_table
from pathlib import Path

PROCESSED = Path('../data/processed')
REPORTS = Path('../reports')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120

---
## 1. Pull CPI Data from BLS

Series: `CUUR0300SA0` — CPI-U, South Region, All Items (not seasonally adjusted).
This is the most relevant inflation measure for Texas workers.

In [ ]:
# Fetch South Region CPI-U monthly data
cpi_monthly = fetch_cpi_series('CUUR0300SA0', start_year=2010, end_year=2025)
cpi_yearly = annual_cpi(cpi_monthly)

print(f"CPI data: {len(cpi_monthly)} monthly observations, {len(cpi_yearly)} years")
print(f"\nAnnual CPI-U (South Region):")
for _, row in cpi_yearly.iterrows():
    print(f"  {int(row['year'])}: {row['annual_cpi']:.1f}")

---
## 2. The Erosion: $10.60 Frozen in Time

The $10.60 wage assumption has been in HHSC's rate methodology for years.
We'll show what happens when a wage is frozen while prices rise.

We test from 2015 (conservative estimate of when $10.60 was locked in).
If the researcher finds the actual year, we'll update.

In [ ]:
# Build erosion table — what is $10.60 worth in real terms each year?
HHSC_WAGE = 10.60
SET_YEAR = 2015  # Conservative estimate; may be earlier

erosion = build_erosion_table(HHSC_WAGE, SET_YEAR, cpi_yearly)

print(f"=== Purchasing Power Erosion of ${HHSC_WAGE}/hr (set ~{SET_YEAR}) ===")
print(f"{'Year':<6} {'Nominal':>10} {'Real Value':>12} {'Lost':>8}")
print('-' * 40)
for _, row in erosion.iterrows():
    print(f"{int(row['year']):<6} ${row['nominal']:>8.2f}  ${row['real_value']:>10.2f}  "
          f"{row['purchasing_power_lost_pct']:>6.1f}%")

latest = erosion.iloc[-1]
print(f"\n→ In {SET_YEAR} dollars, today's ${HHSC_WAGE} is worth ${latest['real_value']:.2f}")
print(f"→ Workers have lost {latest['purchasing_power_lost_pct']:.1f}% of purchasing power")
print(f"→ That's a silent pay CUT of ${HHSC_WAGE - latest['real_value']:.2f}/hr")

In [ ]:
# Chart: Real wage erosion over time
fig, ax = plt.subplots(figsize=(14, 7))

years = erosion['year'].astype(int)

# Nominal wage (flat line)
ax.plot(years, erosion['nominal'], 'o-', color='#d32f2f', linewidth=3, markersize=8,
        label=f'Nominal wage: ${HHSC_WAGE}/hr (frozen)', zorder=5)

# Real purchasing power (declining)
ax.plot(years, erosion['real_value'], 's-', color='#1976d2', linewidth=3, markersize=8,
        label=f'Real value ({SET_YEAR} dollars)', zorder=5)

# Fill the gap — this is the invisible pay cut
ax.fill_between(years, erosion['real_value'], erosion['nominal'],
                alpha=0.2, color='#d32f2f', label='Purchasing power lost')

# What $10.60 SHOULD be if it kept up with inflation
base_cpi = cpi_yearly.loc[cpi_yearly['year'] == SET_YEAR, 'annual_cpi'].iloc[0]
inflation_adjusted = HHSC_WAGE * (cpi_yearly['annual_cpi'] / base_cpi)
cpi_years = cpi_yearly[cpi_yearly['year'] >= SET_YEAR]
inflation_adj_series = HHSC_WAGE * (cpi_years['annual_cpi'].values / base_cpi)
ax.plot(cpi_years['year'].astype(int), inflation_adj_series, '^--', color='#388e3c',
        linewidth=2, markersize=7, alpha=0.8,
        label=f'What ${HHSC_WAGE} should be (inflation-adjusted)')

# Annotate the current gap
latest_year = int(latest['year'])
latest_real = latest['real_value']
latest_should_be = inflation_adj_series[-1]
ax.annotate(f'Real value: ${latest_real:.2f}\n({latest["purchasing_power_lost_pct"]:.0f}% erosion)',
            xy=(latest_year, latest_real),
            xytext=(latest_year - 2, latest_real - 1.2),
            arrowprops=dict(arrowstyle='->', color='#1976d2', lw=2),
            fontsize=12, fontweight='bold', color='#1976d2')

ax.annotate(f'Should be: ${latest_should_be:.2f}',
            xy=(latest_year, latest_should_be),
            xytext=(latest_year - 2, latest_should_be + 0.8),
            arrowprops=dict(arrowstyle='->', color='#388e3c', lw=2),
            fontsize=12, fontweight='bold', color='#388e3c')

# Reference lines
ax.axhline(y=7.25, color='gray', linestyle=':', alpha=0.5)
ax.text(years.iloc[0], 7.35, 'TX minimum wage: $7.25', fontsize=9, color='gray')

ax.set_xlabel('Year', fontsize=13)
ax.set_ylabel('Hourly Wage ($)', fontsize=13)
ax.set_title('The Invisible Pay Cut: HHSC\'s $10.60 Wage Assumption\n'
             'Frozen While Inflation Erodes Purchasing Power',
             fontweight='bold', fontsize=14)
ax.legend(fontsize=11, loc='center left')
ax.set_ylim(6, max(latest_should_be, HHSC_WAGE) + 3)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:.2f}'))

plt.tight_layout()
plt.savefig(REPORTS / 'real_wage_erosion.png', bbox_inches='tight')
plt.show()

print(f"\nSaved: reports/real_wage_erosion.png")

In [ ]:
# What would $10.60 need to be today to have the same purchasing power?
should_be = deflate_wage(HHSC_WAGE, SET_YEAR, int(cpi_years['year'].iloc[-1]), cpi_yearly)

print(f"To have the same purchasing power as ${HHSC_WAGE} in {SET_YEAR}:")
print(f"  Today's wage should be: ${should_be:.2f}/hr")
print(f"  The $13 'raise' doesn't even match inflation from {SET_YEAR}.")
print(f"  Gap: ${should_be - 13.00:.2f}/hr SHORT of inflation-adjusted parity")
print(f"")
print(f"Annual impact per worker:")
print(f"  Should earn: ${should_be * 2080:,.0f}/yr")
print(f"  Actually earns at $10.60: ${10.60 * 2080:,.0f}/yr")
print(f"  Inflation theft: ${(should_be - 10.60) * 2080:,.0f}/yr per worker")

---
## 3. The Worker Subsidy: $4.9 Billion in Foregone Wages

Direct care workers accept below-market wages because:
1. They care about the people they serve
2. There aren't enough alternative jobs in their area
3. They don't know what they're worth

The gap between what they earn and what the market pays for comparable work
is a **subsidy** — workers are effectively donating money to the Medicaid system.

In [ ]:
# Load existing data
df_bls = pd.read_csv(PROCESSED / 'bls_oews_texas_2024.csv')
df_market = pd.read_csv(PROCESSED / 'market_wages.csv')

# Key numbers from BLS
hha = df_bls[df_bls['soc_code'] == '31-1120'].set_index('measure')['value']
tx_hha_employment = hha['employment']  # 314,610
tx_hha_mean = hha['hourly_mean']  # $12.19
hours_per_year = 2080  # standard full-time

# Comparison wages
comparisons = {
    "Buc-ee's ($18.00)": 18.00,
    'Amazon ($17.50)': 17.50,
    'H-E-B ($15.00)': 15.00,
    'Walmart ($14.00)': 14.00,
    'Whataburger ($13.00)': 13.00,
}

# Current effective wage for HHSC-funded workers
actual_wage = 10.60  # use HHSC assumption as baseline

print('=== The Worker Subsidy: Foregone Wages ===')
print(f'Texas Home Health/Personal Care Aides: {tx_hha_employment:,.0f} workers')
print(f'HHSC target wage: ${actual_wage}/hr')
print(f'Standard year: {hours_per_year} hours')
print()

subsidy_data = []
for label, comp_wage in comparisons.items():
    gap = comp_wage - actual_wage
    per_worker = gap * hours_per_year
    statewide = per_worker * tx_hha_employment
    subsidy_data.append({
        'comparison': label,
        'comp_wage': comp_wage,
        'gap': gap,
        'per_worker_annual': per_worker,
        'statewide_annual': statewide,
    })
    print(f'  vs {label}:')
    print(f'    Gap: ${gap:.2f}/hr × {hours_per_year} hrs = ${per_worker:,.0f}/yr per worker')
    print(f'    Statewide: ${per_worker:,.0f} × {tx_hha_employment:,.0f} = ${statewide/1e9:.1f}B/yr')
    print()

df_subsidy = pd.DataFrame(subsidy_data)

In [ ]:
# Chart: Worker subsidy by comparison employer
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# LEFT: Per-worker annual opportunity cost
colors_bar = ['#d32f2f', '#f57c00', '#1976d2', '#388e3c', '#9c27b0']
bars = axes[0].barh(df_subsidy['comparison'], df_subsidy['per_worker_annual'],
                     color=colors_bar, edgecolor='white', height=0.6)

for bar, val in zip(bars, df_subsidy['per_worker_annual']):
    axes[0].text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
                 f'${val:,.0f}/yr', va='center', fontweight='bold', fontsize=12)

axes[0].set_xlabel('Annual Foregone Wages Per Worker ($)', fontsize=12)
axes[0].set_title('What Each Care Worker Gives Up\nby Staying at $10.60/hr',
                   fontweight='bold', fontsize=14)
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# RIGHT: Statewide subsidy (in billions)
bars2 = axes[1].barh(df_subsidy['comparison'], df_subsidy['statewide_annual'] / 1e9,
                      color=colors_bar, edgecolor='white', height=0.6)

for bar, val in zip(bars2, df_subsidy['statewide_annual'] / 1e9):
    axes[1].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                 f'${val:.1f}B', va='center', fontweight='bold', fontsize=12)

axes[1].set_xlabel('Annual Statewide Subsidy ($ Billions)', fontsize=12)
axes[1].set_title(f'Statewide "Subsidy" from {tx_hha_employment:,.0f} Workers\n'
                   'Accepting Below-Market Wages',
                   fontweight='bold', fontsize=14)

plt.suptitle('Texas Direct Care Workers Subsidize the Medicaid System\n'
             'with Billions in Below-Market Labor Every Year',
             fontweight='bold', fontsize=16, y=1.02)

plt.tight_layout()
plt.savefig(REPORTS / 'worker_subsidy.png', bbox_inches='tight')
plt.show()

print(f"\nSaved: reports/worker_subsidy.png")

---
## 4. Lifetime Earnings: The Career Cost of Care Work

What does choosing direct care over retail cost someone over 10, 20, 30 years?

In [ ]:
# Lifetime earnings comparison
career_years = np.arange(1, 31)
annual_raise = 0.02  # assume 2% annual raise for all paths (conservative)

careers = {
    'Direct Care ($10.60)': 10.60,
    'HHSC "Raise" ($13.00)': 13.00,
    'Walmart ($14.00)': 14.00,
    'H-E-B ($15.00)': 15.00,
    "Buc-ee's ($18.00)": 18.00,
}

fig, ax = plt.subplots(figsize=(14, 8))

career_colors = ['#d32f2f', '#f57c00', '#757575', '#1976d2', '#388e3c']

for (label, start_wage), color in zip(careers.items(), career_colors):
    cumulative = []
    total = 0
    for yr in career_years:
        annual = start_wage * hours_per_year * (1 + annual_raise) ** (yr - 1)
        total += annual
        cumulative.append(total)
    
    ax.plot(career_years, np.array(cumulative) / 1000, '-', linewidth=3,
            color=color, label=label, marker='o' if yr % 5 == 0 else None, markersize=6)
    
    # Label endpoint
    ax.text(30.3, cumulative[-1] / 1000, f'${cumulative[-1]/1000:.0f}K',
            fontweight='bold', fontsize=11, color=color, va='center')

# Highlight the 10, 20, 30 year marks
for yr_mark in [10, 20, 30]:
    ax.axvline(x=yr_mark, color='gray', linestyle=':', alpha=0.3)

# Gap annotation at 30 years
care_30yr = 10.60 * hours_per_year * sum((1 + annual_raise) ** (y - 1) for y in range(1, 31))
bucees_30yr = 18.00 * hours_per_year * sum((1 + annual_raise) ** (y - 1) for y in range(1, 31))
gap_30yr = bucees_30yr - care_30yr

ax.annotate(f'30-year gap: ${gap_30yr/1000:,.0f}K',
            xy=(25, (care_30yr + bucees_30yr) / 2 / 1000),
            fontsize=13, fontweight='bold', color='#d32f2f',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.8))

ax.set_xlabel('Years in Career', fontsize=13)
ax.set_ylabel('Cumulative Earnings ($K)', fontsize=13)
ax.set_title('Lifetime Earnings: Direct Care vs. Market Alternatives\n'
             'Assumes 2% annual raises for all paths',
             fontweight='bold', fontsize=14)
ax.legend(fontsize=11, loc='upper left')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}K'))
ax.set_xlim(0, 33)

plt.tight_layout()
plt.savefig(REPORTS / 'lifetime_earnings_gap.png', bbox_inches='tight')
plt.show()

print(f"\n30-year cumulative earnings:")
for (label, start_wage) in careers.items():
    total = start_wage * hours_per_year * sum((1 + annual_raise) ** (y - 1) for y in range(1, 31))
    print(f"  {label:<30} ${total:>12,.0f}")
print(f"\n  30-year gap (care vs Buc-ee's): ${gap_30yr:,.0f}")

---
## 5. The Full Picture: Wage vs CPI vs Market

Combined chart showing the HHSC assumption, inflation, and where the market actually is.

In [ ]:
# Combined wage vs CPI vs market chart
fig, ax = plt.subplots(figsize=(14, 8))

# HHSC wage assumption (flat)
plot_years = cpi_years['year'].astype(int)
ax.plot(plot_years, [HHSC_WAGE] * len(plot_years), 'o-', color='#d32f2f',
        linewidth=3, markersize=6, label=f'HHSC wage assumption: ${HHSC_WAGE}', zorder=5)

# What $10.60 should be (inflation-adjusted)
ax.plot(plot_years, inflation_adj_series, '^--', color='#388e3c',
        linewidth=2, markersize=6, label='$10.60 inflation-adjusted (CPI-U South)', zorder=4)

# Real purchasing power of $10.60
ax.plot(erosion['year'].astype(int), erosion['real_value'], 's--', color='#1976d2',
        linewidth=2, markersize=6, label=f'Real value of ${HHSC_WAGE} ({SET_YEAR} dollars)', zorder=4)

# Market reference lines
ax.axhline(y=18.00, color='#388e3c', linestyle=':', alpha=0.4)
ax.text(plot_years.iloc[0], 18.15, "Buc-ee's: $18.00", fontsize=9, color='#388e3c')

ax.axhline(y=13.00, color='#f57c00', linestyle=':', alpha=0.4)
ax.text(plot_years.iloc[0], 13.15, 'HHSC "new" target: $13.00', fontsize=9, color='#f57c00')

ax.axhline(y=7.25, color='gray', linestyle=':', alpha=0.3)
ax.text(plot_years.iloc[0], 7.35, 'TX minimum: $7.25', fontsize=9, color='gray')

# BLS 2024 actual marker
ax.plot(2024, 12.19, 'D', color='#9c27b0', markersize=12, zorder=6,
        label='BLS TX HHA mean: $12.19 (May 2024)')

# Shade the gap between nominal and inflation-adjusted
ax.fill_between(plot_years, [HHSC_WAGE] * len(plot_years), inflation_adj_series,
                alpha=0.15, color='#d32f2f', label='Gap: frozen wage vs. inflation')

ax.set_xlabel('Year', fontsize=13)
ax.set_ylabel('Hourly Wage ($)', fontsize=13)
ax.set_title('HHSC Wage Assumption vs. Inflation vs. Market Reality\n'
             'The gap between what the state assumes and what workers need keeps growing',
             fontweight='bold', fontsize=14)
ax.legend(fontsize=10, loc='upper left')
ax.set_ylim(6, 20)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:.2f}'))

plt.tight_layout()
plt.savefig(REPORTS / 'wage_vs_cpi_vs_market.png', bbox_inches='tight')
plt.show()

print(f"Saved: reports/wage_vs_cpi_vs_market.png")

---
## 6. Key Findings

1. **$10.60 has lost 26% of purchasing power** since ~2015 — in real terms it's worth $7.82
2. **Even the $13 "raise" doesn't keep pace with inflation** — should be $14.37 to match 2015 purchasing power
3. **Workers collectively subsidize TX Medicaid by ~$4.8B/yr** vs. Buc-ee's wages
4. **A 30-year career in care work costs ~$624K** vs. retail — that's a house, retirement, kids' college
5. **This is a policy choice, not an economic inevitability** — the state literally publishes the spreadsheet

In [ ]:
# Citation-ready stats
print('=== Citation-Ready Stats for Notebook 05 ===')
print(f'CPI data: BLS CPI-U South Region (CUUR0300SA0), {SET_YEAR}-{int(cpi_years["year"].iloc[-1])}')
print(f'HHSC wage assumption: ${HHSC_WAGE}/hr (PFD Wage Calculator cell B7, updated 2/7/2025)')
print(f'Assumed set year: {SET_YEAR} (conservative)')
print(f'Real value of ${HHSC_WAGE} today: ${latest["real_value"]:.2f} ({SET_YEAR} dollars)')
print(f'Purchasing power lost: {latest["purchasing_power_lost_pct"]:.1f}%')
print(f'Inflation-adjusted value should be: ${should_be:.2f}/hr')
print(f'')
print(f'TX HHA employment: {tx_hha_employment:,.0f} (BLS OEWS May 2024)')
print(f'Per-worker subsidy vs Buc-ee\'s: ${(18.00-10.60)*2080:,.0f}/yr')
print(f'Statewide subsidy vs Buc-ee\'s: ${(18.00-10.60)*2080*tx_hha_employment/1e9:.1f}B/yr')
print(f'')
print(f'30-year earnings gap (care vs Buc-ee\'s): ${gap_30yr:,.0f}')
print(f'All charts saved to reports/')